# Глава 3 — Faster R-CNN (task_09)

torchvision `fasterrcnn_resnet50_fpn_v2` на 4 вариантах: `baseline` → `aug_geom` → `aug_oversample` → `aug_diffusion`.

Скрипт: `chapter3_torchvision_runner.py --variant all`. Лог: `/tmp/train_faster_rcnn.log`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import time
from IPython.display import Image, display, clear_output

ROOT = Path('/home/vanusha/diplom/diploma-plant-disease')
TASK_DIR = ROOT / 'code/results/task_09'
VARIANTS = ['baseline', 'aug_geom', 'aug_oversample', 'aug_diffusion']
LOG_FILE = Path('/tmp/train_faster_rcnn.log')

## 🔴 Live monitor (auto-refresh 10s)

In [ ]:
try:
    while True:
        clear_output(wait=True)
        now = time.strftime('%H:%M:%S')
        print(f'[live {now}]')
        if LOG_FILE.exists():
            lines = [l for l in LOG_FILE.read_text(errors='ignore').splitlines() if l.strip()]
            print('\n'.join(lines[-25:]))
        else:
            print('не запущено')
        fig, axs = plt.subplots(1, 2, figsize=(12, 4))
        any_data = False
        for v in VARIANTS:
            p = TASK_DIR / f'faster_rcnn_{v}' / 'metrics.csv'
            if not p.exists():
                continue
            try:
                df = pd.read_csv(p)
                if 'val_mAP50' in df.columns:
                    axs[0].plot(df['epoch'], df['val_mAP50'], label=v, marker='o', markersize=3)
                if 'val_mAP5095' in df.columns:
                    axs[1].plot(df['epoch'], df['val_mAP5095'], label=v, marker='o', markersize=3)
                any_data = True
            except Exception:
                pass
        if any_data:
            axs[0].set_title('val mAP@50'); axs[1].set_title('val mAP@50-95')
            for ax in axs: ax.set_xlabel('epoch'); ax.grid(True, alpha=0.3); ax.legend()
            plt.tight_layout(); plt.show()
        else:
            plt.close(fig)
        time.sleep(10)
except KeyboardInterrupt:
    print('\n[monitor stopped]')

## Сводная таблица

In [ ]:
p = TASK_DIR / 'summary.csv'
if p.exists():
    display(pd.read_csv(p))
else:
    print('ещё нет')

## Learning curves / Confusion matrix

In [ ]:
for v in VARIANTS:
    lc = TASK_DIR / f'faster_rcnn_{v}' / 'learning_curves.png'
    cm = TASK_DIR / f'faster_rcnn_{v}' / 'confusion_matrix.png'
    print(f'=== {v} ===')
    if lc.exists(): display(Image(str(lc)))
    if cm.exists(): display(Image(str(cm)))